In [1]:
!pip install -q -U wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 62.9 MB/s eta 0:00:00


In [2]:
%%writefile rectified_flow_ct_augmentation.py
"""
=============================================================================
 Class-Conditional Rectified Flow for CT Scan Augmentation
 Dataset: IQ-OTH/NCCD Lung Cancer CT
=============================================================================

 This is the SAME technique used by FLUX, Stable Diffusion 3, and Seedream.
 Same U-Net architecture as DDPM, but with straight-line flow matching.

 Usage:
   Multi-GPU:  torchrun --nproc_per_node=2 rectified_flow_ct_augmentation.py
   Single-GPU: python rectified_flow_ct_augmentation.py
"""

import os
import sys
import math
import copy
import time
import random
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import WeightedRandomSampler
from sklearn.model_selection import train_test_split

try:
    import wandb
    HAS_WANDB = True
except ImportError:
    HAS_WANDB = False

try:
    import albumentations as A
    HAS_ALBUM = True
except ImportError:
    HAS_ALBUM = False

import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

TRAINING_START_TIME = time.time()






# ═══════════════════════════════════════════════════════════════════════
# DDP SETUP (identical to DDPM)
# ═══════════════════════════════════════════════════════════════════════

def setup_distributed():
    """Initialize DDP if launched with torchrun, else single-GPU."""
    if "RANK" in os.environ:
        rank = int(os.environ["RANK"])
        local_rank = int(os.environ["LOCAL_RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        torch.cuda.set_device(local_rank)
        dist.init_process_group("nccl", rank=rank, world_size=world_size,
                                device_id=torch.device('cuda', local_rank))
        return rank, local_rank, world_size
    else:
        return 0, 0, 1


def cleanup_distributed():
    if dist.is_initialized():
        dist.destroy_process_group()


def is_main_process():
    return not dist.is_initialized() or dist.get_rank() == 0


# Initialize
RANK, LOCAL_RANK, WORLD_SIZE = setup_distributed()
IS_DDP = WORLD_SIZE > 1


# ═══════════════════════════════════════════════════════════════════════
# GPU AUTO-DETECTION
# ═══════════════════════════════════════════════════════════════════════

def detect_gpu_config(local_rank=0):
    """Auto-detect GPU capabilities and set optimal precision + batch size."""
    if not torch.cuda.is_available():
        return {
            'device': torch.device('cpu'), 'precision': 'fp32',
            'amp_dtype': torch.float32, 'use_amp': False,
            'batch_size': 4, 'compile': False, 'gpu_name': 'CPU',
            'vram_gb': 0, 'compute_cap': '0.0', 'num_workers': 2,
        }

    gpu_name = torch.cuda.get_device_name(local_rank)
    vram_gb = torch.cuda.get_device_properties(local_rank).total_memory / 1024**3
    major, minor = torch.cuda.get_device_capability(local_rank)

    supports_bf16 = major >= 8
    supports_compile = major >= 7

    amp_dtype = torch.bfloat16 if supports_bf16 else torch.float16
    precision = 'bf16' if supports_bf16 else 'fp16'

    # Auto batch size — 224×224 uses ~3× more VRAM than 128×128
    if vram_gb >= 70:
        batch_size = 64
    elif vram_gb >= 35:
        batch_size = 32
    elif vram_gb >= 20:
        batch_size = 24
    elif vram_gb >= 14:
        batch_size = 12   # T4 16GB — 224×224 needs more VRAM than 128
    elif vram_gb >= 10:
        batch_size = 6
    else:
        batch_size = 4

    return {
        'device': torch.device('cuda', local_rank),
        'precision': precision,
        'amp_dtype': amp_dtype,
        'use_amp': True,
        'batch_size': batch_size,
        'compile': supports_compile,
        'gpu_name': gpu_name,
        'vram_gb': vram_gb,
        'compute_cap': f"{major}.{minor}",
        'num_workers': 4 if vram_gb >= 35 else 2,
    }


gpu_cfg = detect_gpu_config(local_rank=LOCAL_RANK)


# ═══════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════

class Config:
    # ── Dataset ──
    DATASET_ROOT = "/kaggle/input/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset"
    OUTPUT_DIR = "/kaggle/working/rf_output"
    SYNTHETIC_DIR = "/kaggle/working/synthetic_images_rf"

    # ── Classes ──
    CLASSES = ["Bengin cases", "Malignant cases", "Normal cases"]
    CLASS_TO_IDX = {"Bengin cases": 0, "Malignant cases": 1, "Normal cases": 2}
    IDX_TO_CLASS = {0: "Benign", 1: "Malignant", 2: "Normal"}
    NUM_CLASSES = 3

    # ── Image ──
    IMG_SIZE = 224              # ← Upgraded from 128 for better quality
    IMG_CHANNELS = 1            # Grayscale CT scans

    # ── Rectified Flow (replaces DDPM beta schedule) ──
    # No beta_start / beta_end needed — RF uses t ∈ [0, 1] directly
    SIGMA_MIN = 1e-4            # Small offset to avoid division by zero at t=0
    TIMESTEP_SAMPLING = "logit_normal"   # "uniform" or "logit_normal" (SD3-style)
    LOGIT_NORMAL_MEAN = 0.0     # Center of logit-normal distribution
    LOGIT_NORMAL_STD = 1.0      # Spread — emphasizes mid-timesteps

    # ── Model (U-Net — same architecture as DDPM) ──
    BASE_CHANNELS = 128         # Channel progression: 128→256→512→1024
    TIME_EMB_DIM = 256          # Timestep embedding dimension
    CLASS_EMB_DIM = 256         # Class embedding dimension

    # ── Training ──
    EPOCHS = 3000
    BATCH_SIZE = gpu_cfg['batch_size']
    LEARNING_RATE = 1.5e-4 * WORLD_SIZE
    EMA_DECAY = 0.9999
    SAVE_EVERY = 20             # Save latest_checkpoint.pt every N epochs (overwrite)
    SAMPLE_EVERY = 25

    # ── Mixed Precision ──
    USE_AMP = gpu_cfg['use_amp']
    AMP_DTYPE = gpu_cfg['amp_dtype']
    USE_COMPILE = gpu_cfg['compile']

    # ── Distributed ──
    WORLD_SIZE = WORLD_SIZE
    RANK = RANK
    LOCAL_RANK = LOCAL_RANK
    IS_DDP = IS_DDP

    # ── W&B ──
    WANDB_PROJECT = "ct-rectified-flow"
    WANDB_RUN_NAME = f"rf-{WORLD_SIZE}xGPU-ddp" if IS_DDP else "rf-1xGPU"
    USE_WANDB = True

    # ── Kaggle Time Guard ──
    MAX_TRAINING_HOURS = 10.0
    GENERATION_BUDGET_MIN = 30

    # ── Resume ──
    RESUME_CHECKPOINT = None    # Set to checkpoint path to resume

    # ── Generation ──
    NUM_SYNTHETIC_BENIGN = 300
    NUM_SYNTHETIC_NORMAL = 100
    EULER_STEPS = 50            # ODE solver steps (more = cleaner images)
    CFG_SCALE = 4.0             # Classifier-free guidance scale

    # ── Reproducibility ──
    SEED = 42

    # ── Device ──
    DEVICE = gpu_cfg['device']
    NUM_WORKERS = gpu_cfg['num_workers']


cfg = Config()


# ═══════════════════════════════════════════════════════════════════════
# PART 2: RECTIFIED FLOW SCHEDULER
# ═══════════════════════════════════════════════════════════════════════




class RectifiedFlowScheduler:
    """
    Rectified Flow noise scheduler.
    
    Key differences from DDPM DiffusionScheduler:
    ─────────────────────────────────────────────────────────────────
    │ Aspect              │ DDPM                    │ Rectified Flow │
    │─────────────────────│─────────────────────────│────────────────│
    │ Time range          │ t ∈ {0, ..., T-1}       │ t ∈ [0, 1]     │
    │ Forward process     │ x_t = √ᾱ·x0 + √(1-ᾱ)·ε │ x_t = (1-t)·x0 + t·ε │
    │ Model predicts      │ noise ε                 │ velocity v = ε - x0   │
    │ Training loss       │ MSE(ε̂, ε)               │ MSE(v̂, ε - x0)       │
    │ Sampling            │ DDPM/DDIM reverse        │ Euler ODE solver      │
    │ Pre-computed tables │ betas, alphas_cumprod... │ NONE needed!          │
    ─────────────────────────────────────────────────────────────────
    """

    def __init__(self, sigma_min=1e-4, device='cuda'):
        self.sigma_min = sigma_min
        self.device = device

    def sample_timesteps(self, batch_size, method="logit_normal", mean=0.0, std=1.0):
        """
        Sample timesteps t ∈ [sigma_min, 1] for training.
        
        Args:
            batch_size: Number of timesteps to sample.
            method: "uniform" or "logit_normal"
              - uniform: t ~ U[sigma_min, 1]
              - logit_normal: t = sigmoid(N(mean, std²))
                This is SD3's technique — emphasizes mid-timesteps (t≈0.5)
                where the model has the hardest learning signal.
        
        Returns:
            t: (B,) tensor of timesteps in [sigma_min, 1]
        """
        if method == "logit_normal":
            # Sample from logit-normal distribution
            # t = sigmoid(z), z ~ N(mean, std²)
            z = torch.randn(batch_size, device=self.device) * std + mean
            t = torch.sigmoid(z)
            # Clamp to [sigma_min, 1 - sigma_min] for numerical stability
            t = t.clamp(self.sigma_min, 1.0 - self.sigma_min)
        else:
            # Simple uniform sampling
            t = torch.rand(batch_size, device=self.device)
            t = t * (1.0 - self.sigma_min) + self.sigma_min  # [sigma_min, 1]

        return t

    def add_noise(self, x0, t, noise=None):
        """
        Forward process: interpolate between clean image and noise.
        
        DDPM:  x_t = √ᾱ_t · x_0 + √(1-ᾱ_t) · ε       (curved)
        RF:    x_t = (1-t) · x_0 + t · ε                (straight!)
        
        Args:
            x0: (B, C, H, W) clean images in [-1, 1]
            t:  (B,) timesteps in [0, 1]
            noise: (B, C, H, W) optional pre-generated noise
            
        Returns:
            x_t: (B, C, H, W) noisy image
            noise: (B, C, H, W) the noise used
            velocity: (B, C, H, W) target velocity v = noise - x0
        """
        if noise is None:
            noise = torch.randn_like(x0)

        # Reshape t for broadcasting: (B,) → (B, 1, 1, 1)
        t_view = t.view(-1, 1, 1, 1)

        # Straight-line interpolation (this is the ENTIRE math change from DDPM)
        x_t = (1.0 - t_view) * x0 + t_view * noise

        # Target: velocity field v = noise - x0
        velocity = noise - x0

        return x_t, noise, velocity

    @torch.no_grad()
    def sample_euler(self, model, shape, class_labels, num_steps=30,
                     cfg_scale=3.0, num_classes=3):
        """
        Generate images using Euler ODE solver.
        
        Starting from pure noise (t=1), step backwards to clean image (t=0)
        using the predicted velocity field.
        
        This replaces DDPM's reverse sampling and DDIM.
        
        Algorithm:
            x_1 = noise ~ N(0, I)
            for t from 1 to 0 in num_steps:
                v = model(x_t, t, class)       # predict velocity
                x_{t-dt} = x_t - dt * v        # Euler step
            return x_0
        
        Args:
            model: velocity prediction network (U-Net)
            shape: (B, C, H, W) output shape
            class_labels: (B,) class indices
            num_steps: number of Euler integration steps
            cfg_scale: classifier-free guidance scale
            num_classes: number of classes (for unconditional label)
            
        Returns:
            x_0: (B, C, H, W) generated images in [-1, 1]
        """
        model.eval()
        b = shape[0]
        device = next(model.parameters()).device

        # Start from pure noise at t=1
        x = torch.randn(shape, device=device)

        # Uniform timestep schedule from 1 → sigma_min (NOT 0, to avoid numerical instability)
        timesteps = torch.linspace(1.0, self.sigma_min, num_steps + 1, device=device)

        for i in tqdm(range(num_steps), desc="RF Euler Sampling", disable=not True):
            t_current = timesteps[i]
            t_next = timesteps[i + 1]
            dt = t_current - t_next  # Adaptive step size
            t_batch = torch.full((b,), t_current, device=device)

            # ── Classifier-Free Guidance ──
            # Conditional prediction
            v_cond = model(x, t_batch, class_labels)
            # Unconditional prediction
            uncond_labels = torch.full_like(class_labels, num_classes)
            v_uncond = model(x, t_batch, uncond_labels)
            # Guided velocity
            v = v_uncond + cfg_scale * (v_cond - v_uncond)

            # ── Euler step: x_{t-dt} = x_t - dt * v ──
            x = x - dt * v

        # Clamp to valid range
        x = torch.clamp(x, -1.0, 1.0)
        return x

    @torch.no_grad()
    def sample_midpoint(self, model, shape, class_labels, num_steps=20,
                        cfg_scale=3.0, num_classes=3):
        """
        Generate images using Midpoint method (2nd-order ODE solver).
        Better quality than Euler at the same number of steps, at 2× compute cost.
        
        Algorithm:
            x_1 = noise ~ N(0, I)
            for t from 1 to 0:
                v1 = model(x_t, t)               # velocity at current point
                x_mid = x_t - (dt/2) * v1        # half step
                v2 = model(x_mid, t - dt/2)      # velocity at midpoint
                x_{t-dt} = x_t - dt * v2         # full step with midpoint velocity
        """
        model.eval()
        b = shape[0]
        device = next(model.parameters()).device

        x = torch.randn(shape, device=device)
        # Stop at sigma_min instead of 0 to avoid numerical instability
        timesteps = torch.linspace(1.0, self.sigma_min, num_steps + 1, device=device)

        for i in tqdm(range(num_steps), desc="RF Midpoint Sampling", disable=not True):
            t_current = timesteps[i]
            t_next = timesteps[i + 1]
            dt = t_current - t_next  # Adaptive step size
            t_mid = t_current - dt / 2

            # ── Step 1: velocity at current point ──
            t_batch = torch.full((b,), t_current, device=device)
            v_cond = model(x, t_batch, class_labels)
            uncond_labels = torch.full_like(class_labels, num_classes)
            v_uncond = model(x, t_batch, uncond_labels)
            v1 = v_uncond + cfg_scale * (v_cond - v_uncond)

            # ── Step 2: velocity at midpoint ──
            x_mid = x - (dt / 2) * v1
            t_mid_batch = torch.full((b,), max(t_mid.item(), self.sigma_min), device=device)
            v_cond_mid = model(x_mid, t_mid_batch, class_labels)
            v_uncond_mid = model(x_mid, t_mid_batch, uncond_labels)
            v2 = v_uncond_mid + cfg_scale * (v_cond_mid - v_uncond_mid)

            # ── Full step with midpoint velocity ──
            x = x - dt * v2

        x = torch.clamp(x, -1.0, 1.0)
        return x


# ═══════════════════════════════════════════════════════════════════════
# PART 3: U-NET MODEL (VELOCITY PREDICTION) — Building Blocks
# ═══════════════════════════════════════════════════════════════════════

class SinusoidalPositionEmbedding(nn.Module):
    """
    Timestep embedding using sinusoidal functions (like Transformers).
    
    For Rectified Flow, t is continuous in [0, 1].
    We scale it to [0, 1000] range before embedding for numerical stability
    (matching the DDPM embedding range).
    """

    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        """
        Args:
            t: (B,) timesteps, continuous in [0, 1]
        Returns:
            emb: (B, dim) sinusoidal embeddings
        """
        device = t.device
        # Scale to [0, 1000] range for numerical stability
        t_scaled = t.float() * 1000.0

        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t_scaled[:, None] * emb[None, :]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb


class SelfAttention(nn.Module):
    """
    Self-attention for spatial feature maps.
    Uses F.scaled_dot_product_attention for Flash Attention support (PyTorch 2.0+).
    """

    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.q = nn.Conv2d(channels, channels, 1)
        self.k = nn.Conv2d(channels, channels, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)

        q = self.q(h).view(B, C, H * W).permute(0, 2, 1)  # (B, HW, C)
        k = self.k(h).view(B, C, H * W).permute(0, 2, 1)  # (B, HW, C)
        v = self.v(h).view(B, C, H * W).permute(0, 2, 1)  # (B, HW, C)

        # Use PyTorch 2.0+ SDPA for Flash Attention when available
        out = F.scaled_dot_product_attention(q, k, v)       # (B, HW, C)
        out = out.permute(0, 2, 1).view(B, C, H, W)

        return x + self.proj(out)


class ResBlock(nn.Module):
    """Residual block with time and class conditioning."""

    def __init__(self, in_ch, out_ch, time_emb_dim, class_emb_dim, use_attention=False):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.GroupNorm(8, in_ch),
            nn.SiLU(),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
        )

        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_ch),
        )

        self.class_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(class_emb_dim, out_ch),
        )

        self.conv2 = nn.Sequential(
            nn.GroupNorm(8, out_ch),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
        )

        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.attention = SelfAttention(out_ch) if use_attention else nn.Identity()

    def forward(self, x, t_emb, c_emb):
        h = self.conv1(x)
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = h + self.class_mlp(c_emb)[:, :, None, None]
        h = self.conv2(h)
        h = h + self.skip(x)
        h = self.attention(h)
        return h


# ═══════════════════════════════════════════════════════════════════════
# CONDITIONAL U-NET (VELOCITY PREDICTION)
# ═══════════════════════════════════════════════════════════════════════

class ConditionalUNet(nn.Module):
    """
    U-Net for VELOCITY prediction (Rectified Flow).
    
    Architecture identical to the DDPM noise-prediction U-Net.
    The only semantic difference is:
      - DDPM: output = predicted noise ε̂
      - RF:   output = predicted velocity v̂ = ε - x₀
    
    The architecture doesn't need to know — it's the loss function that differs.
    
    Parameters: ~19M with base_channels=64
    """

    def __init__(self, img_channels=1, base_channels=64,
                 time_emb_dim=256, num_classes=3, class_emb_dim=256):
        super().__init__()

        # ── Embeddings ──
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )

        # num_classes + 1 for the "unconditional" class (CFG)
        self.class_embed = nn.Embedding(num_classes + 1, class_emb_dim)

        ch = base_channels  # 64

        # ── Encoder ──
        self.enc_conv0 = nn.Conv2d(img_channels, ch, 3, padding=1)

        self.enc1 = ResBlock(ch, ch, time_emb_dim, class_emb_dim)               # 64
        self.down1 = nn.Conv2d(ch, ch, 4, stride=2, padding=1)                  # /2

        self.enc2 = ResBlock(ch, ch * 2, time_emb_dim, class_emb_dim)           # 128
        self.down2 = nn.Conv2d(ch * 2, ch * 2, 4, stride=2, padding=1)         # /4

        self.enc3 = ResBlock(ch * 2, ch * 4, time_emb_dim, class_emb_dim,
                             use_attention=True)                                 # 256
        self.down3 = nn.Conv2d(ch * 4, ch * 4, 4, stride=2, padding=1)         # /8

        # ── Bottleneck ──
        self.bot1 = ResBlock(ch * 4, ch * 8, time_emb_dim, class_emb_dim,
                             use_attention=True)                                 # 512
        self.bot2 = ResBlock(ch * 8, ch * 8, time_emb_dim, class_emb_dim,
                             use_attention=True)
        self.bot3 = ResBlock(ch * 8, ch * 4, time_emb_dim, class_emb_dim)      # Back to 256

        # ── Decoder ──
        self.up3 = nn.ConvTranspose2d(ch * 4, ch * 4, 4, stride=2, padding=1)   # ×2
        self.dec3 = ResBlock(ch * 8, ch * 2, time_emb_dim, class_emb_dim,
                             use_attention=True)                                  # 128

        self.up2 = nn.ConvTranspose2d(ch * 2, ch * 2, 4, stride=2, padding=1)   # ×2
        self.dec2 = ResBlock(ch * 4, ch, time_emb_dim, class_emb_dim)            # 64

        self.up1 = nn.ConvTranspose2d(ch, ch, 4, stride=2, padding=1)            # ×2
        self.dec1 = ResBlock(ch * 2, ch, time_emb_dim, class_emb_dim)            # 64

        # ── Output ──
        self.out = nn.Sequential(
            nn.GroupNorm(8, ch),
            nn.SiLU(),
            nn.Conv2d(ch, img_channels, 3, padding=1),
        )

    def forward(self, x, t, c):
        """
        Args:
            x: (B, 1, H, W) — noisy image x_t
            t: (B,) — continuous timestep in [0, 1]
            c: (B,) — class label (0=Benign, 1=Malignant, 2=Normal, 3=unconditional)
        
        Returns:
            v: (B, 1, H, W) — predicted velocity (same shape as input)
        """
        # Embeddings
        t_emb = self.time_embed(t)
        c_emb = self.class_embed(c)

        # Encoder
        x0 = self.enc_conv0(x)
        e1 = self.enc1(x0, t_emb, c_emb)
        e1_down = self.down1(e1)

        e2 = self.enc2(e1_down, t_emb, c_emb)
        e2_down = self.down2(e2)

        e3 = self.enc3(e2_down, t_emb, c_emb)
        e3_down = self.down3(e3)

        # Bottleneck
        b = self.bot1(e3_down, t_emb, c_emb)
        b = self.bot2(b, t_emb, c_emb)
        b = self.bot3(b, t_emb, c_emb)

        # Decoder with skip connections
        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3, t_emb, c_emb)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2, t_emb, c_emb)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1, t_emb, c_emb)

        return self.out(d1)


# ═══════════════════════════════════════════════════════════════════════
# PART 4: EXPONENTIAL MOVING AVERAGE (EMA)
# ═══════════════════════════════════════════════════════════════════════


class EMA:
    """
    Exponential Moving Average of model parameters.
    Keeps a smoothed copy of the model for better generation quality.
    """

    def __init__(self, model, decay=0.9999):
        self.decay = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        """Update EMA weights with current model weights."""
        for s_param, m_param in zip(self.shadow.parameters(), model.parameters()):
            s_param.data.mul_(self.decay).add_(m_param.data, alpha=1 - self.decay)

    def forward(self, *args, **kwargs):
        return self.shadow(*args, **kwargs)

    def state_dict(self):
        return self.shadow.state_dict()

    def load_state_dict(self, state_dict):
        self.shadow.load_state_dict(state_dict)


# ═══════════════════════════════════════════════════════════════════════
# PART 5: DATA + TRAINING + GENERATION
# ═══════════════════════════════════════════════════════════════════════








# ═══════════════════════════════════════════════════════════════════════
# SEEDING & CUDA SETUP
# ═══════════════════════════════════════════════════════════════════════

def seed_everything(seed, rank=0):
    random.seed(seed + rank)
    np.random.seed(seed + rank)
    torch.manual_seed(seed + rank)
    torch.cuda.manual_seed_all(seed + rank)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


seed_everything(cfg.SEED, RANK)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# Create output directories (rank 0 only)
if is_main_process():
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    os.makedirs(cfg.SYNTHETIC_DIR, exist_ok=True)
    for cls_name in cfg.IDX_TO_CLASS.values():
        os.makedirs(os.path.join(cfg.SYNTHETIC_DIR, cls_name), exist_ok=True)
if IS_DDP:
    dist.barrier()

if is_main_process():
    print(f"{'='*60}")
    print(f"RECTIFIED FLOW — CT Scan Augmentation")
    print(f"{'='*60}")
    print(f"  GPU:           {gpu_cfg['gpu_name']}")
    if torch.cuda.is_available():
        print(f"  VRAM:          {gpu_cfg['vram_gb']:.1f} GB")
    print(f"  Precision:     {gpu_cfg['precision'].upper()}")
    print(f"  Batch Size:    {cfg.BATCH_SIZE} per GPU × {WORLD_SIZE} = {cfg.BATCH_SIZE * WORLD_SIZE} effective")
    print(f"  Image Size:    {cfg.IMG_SIZE}×{cfg.IMG_SIZE}")
    print(f"  Timestep:      {cfg.TIMESTEP_SAMPLING} sampling")
    print(f"  DDP:           {'✅ ' + str(WORLD_SIZE) + ' GPUs' if IS_DDP else '❌ Single'}")
    print(f"  torch.compile: {'✅' if cfg.USE_COMPILE else '❌'}")
    print(f"{'='*60}")


# ═══════════════════════════════════════════════════════════════════════
# DATA PREPARATION
# ═══════════════════════════════════════════════════════════════════════

def load_dataset_paths(root_dir):
    """Load all image paths and labels from the dataset directory."""
    image_paths, labels = [], []
    for class_name in cfg.CLASSES:
        class_dir = os.path.join(root_dir, class_name)
        if not os.path.exists(class_dir):
            if is_main_process():
                print(f"WARNING: {class_dir} not found!")
            continue
        for img_name in sorted(os.listdir(class_dir)):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
                image_paths.append(os.path.join(class_dir, img_name))
                labels.append(cfg.CLASS_TO_IDX[class_name])
    return image_paths, labels


class CTDataset(Dataset):
    """CT scan dataset with optional augmentation."""

    def __init__(self, image_paths, labels, img_size=224, augment=False):
        self.image_paths = image_paths
        self.labels = labels
        self.img_size = img_size
        self.augment = augment

        if HAS_ALBUM:
            if augment:
                self.transform = A.Compose([
                    A.Resize(img_size, img_size),
                    A.HorizontalFlip(p=0.5),
                    A.Rotate(limit=15, p=0.5, border_mode=0),
                    A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.3, border_mode=0),
                    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
                    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
                    A.GaussNoise(p=0.2),
                ])
            else:
                self.transform = A.Compose([A.Resize(img_size, img_size)])
        else:
            self.transform = None

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('L')
        img = np.array(img)

        if self.transform is not None:
            img = self.transform(image=img)['image']
        else:
            from torchvision.transforms.functional import resize
            img = np.array(resize(Image.fromarray(img), (self.img_size, self.img_size)))

        # Normalize to [-1, 1]
        img = img.astype(np.float32) / 255.0
        img = img * 2.0 - 1.0
        img = torch.FloatTensor(img).unsqueeze(0)  # (1, H, W)
        label = torch.LongTensor([self.labels[idx]]).squeeze()
        return img, label


# Sequential split (files are numbered 1-N, so sequential = no data leakage)
def sequential_split(paths, labels, train_ratio=0.70, val_ratio=0.15):
    """Split by taking first N% per class to avoid data leakage."""
    from collections import defaultdict
    class_files = defaultdict(list)
    for p, l in zip(paths, labels):
        class_files[l].append(p)

    train_p, train_l = [], []
    val_p, val_l = [], []
    test_p, test_l = [], []

    for cls_idx, cls_paths in sorted(class_files.items()):
        n = len(cls_paths)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
        # Files are already sorted by number from load_dataset_paths
        train_p.extend(cls_paths[:n_train])
        train_l.extend([cls_idx] * n_train)
        val_p.extend(cls_paths[n_train:n_train + n_val])
        val_l.extend([cls_idx] * n_val)
        test_p.extend(cls_paths[n_train + n_val:])
        test_l.extend([cls_idx] * (n - n_train - n_val))

    return train_p, val_p, test_p, train_l, val_l, test_l

all_paths, all_labels = load_dataset_paths(cfg.DATASET_ROOT)
if is_main_process():
    print(f"\nTotal images: {len(all_paths)}")
    for idx, count in sorted(Counter(all_labels).items()):
        print(f"  {cfg.IDX_TO_CLASS[idx]}: {count}")

train_paths, val_paths, test_paths, train_labels, val_labels, test_labels = sequential_split(
    all_paths, all_labels, train_ratio=0.70, val_ratio=0.15
)

if is_main_process():
    print(f"Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")
    print(f"  (Sequential split — NO data leakage, only train set used for RF)")

# Create dataset and loader — ONLY train set for RF training
train_dataset = CTDataset(train_paths, train_labels, cfg.IMG_SIZE, augment=True)

if IS_DDP:
    train_sampler = DistributedSampler(train_dataset, num_replicas=WORLD_SIZE, rank=RANK, shuffle=True)
else:
    class_counts = Counter(train_labels)
    class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
    sample_weights = [class_weights[label] for label in train_labels]
    train_sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_labels), replacement=True)

train_loader = DataLoader(
    train_dataset, batch_size=cfg.BATCH_SIZE,
    sampler=train_sampler, num_workers=cfg.NUM_WORKERS,
    pin_memory=True, drop_last=True, persistent_workers=True
)


# ═══════════════════════════════════════════════════════════════════════
# TRAINING FUNCTION
# ═══════════════════════════════════════════════════════════════════════

def train():
    if is_main_process():
        print(f"\n{'='*60}")
        print(f"TRAINING — Rectified Flow")
        print(f"  Precision:     {gpu_cfg['precision'].upper()} AMP")
        print(f"  Batch:         {cfg.BATCH_SIZE} × {WORLD_SIZE} = {cfg.BATCH_SIZE * WORLD_SIZE}")
        print(f"  Resolution:    {cfg.IMG_SIZE}×{cfg.IMG_SIZE}")
        print(f"  Timestep:      {cfg.TIMESTEP_SAMPLING}")
        print(f"  Time Guard:    {cfg.MAX_TRAINING_HOURS}h")
        print(f"{'='*60}")

    # ── W&B Init ──
    if is_main_process() and cfg.USE_WANDB and HAS_WANDB:
        try:
            from kaggle_secrets import UserSecretsClient
            wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
        except Exception:
            wandb.login()

        wandb.init(
            project=cfg.WANDB_PROJECT,
            name=cfg.WANDB_RUN_NAME,
            config={
                "method": "Rectified Flow",
                "model": "ConditionalUNet",
                "img_size": cfg.IMG_SIZE,
                "epochs": cfg.EPOCHS,
                "batch_size_per_gpu": cfg.BATCH_SIZE,
                "effective_batch_size": cfg.BATCH_SIZE * WORLD_SIZE,
                "learning_rate": cfg.LEARNING_RATE,
                "precision": gpu_cfg['precision'],
                "gpu": gpu_cfg['gpu_name'],
                "num_gpus": WORLD_SIZE,
                "timestep_sampling": cfg.TIMESTEP_SAMPLING,
                "euler_steps": cfg.EULER_STEPS,
                "cfg_scale": cfg.CFG_SCALE,
                "ema_decay": cfg.EMA_DECAY,
            },
        )
        print("✅ W&B initialized")

    # ── Model ──
    model = ConditionalUNet(
        img_channels=cfg.IMG_CHANNELS,
        base_channels=cfg.BASE_CHANNELS,
        time_emb_dim=cfg.TIME_EMB_DIM,
        num_classes=cfg.NUM_CLASSES,
        class_emb_dim=cfg.CLASS_EMB_DIM,
    ).to(cfg.DEVICE)

    # torch.compile
    if cfg.USE_COMPILE:
        if is_main_process():
            print("Compiling model with torch.compile...")
        model = torch.compile(model, mode='reduce-overhead')
        if is_main_process():
            print("✅ Compiled")

    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model

    # DDP wrapping
    if IS_DDP:
        model = DDP(model, device_ids=[LOCAL_RANK], output_device=LOCAL_RANK)
        if is_main_process():
            print(f"✅ DDP with {WORLD_SIZE} GPUs")

    total_params = sum(p.numel() for p in raw_model.parameters())
    if is_main_process():
        print(f"Parameters: {total_params:,} ({total_params / 1e6:.2f}M)")

    # ── Scheduler, Optimizer, EMA ──
    scheduler = RectifiedFlowScheduler(sigma_min=cfg.SIGMA_MIN, device=cfg.DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=1e-4)
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)
    ema = EMA(raw_model, decay=cfg.EMA_DECAY)

    # AMP scaler (only for FP16, not BF16)
    use_scaler = cfg.USE_AMP and cfg.AMP_DTYPE == torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)

    # Training state
    train_losses = []
    best_loss = float('inf')
    stop_reason = "completed"
    start_epoch = 1
    p_uncond = 0.1  # CFG label dropout rate

    # ═══════════════════════════════════════════════════════════════
    # RESUME FROM CHECKPOINT
    # ═══════════════════════════════════════════════════════════════
    if cfg.RESUME_CHECKPOINT and os.path.exists(cfg.RESUME_CHECKPOINT):
        if is_main_process():
            print(f"\n🔄 Resuming from: {cfg.RESUME_CHECKPOINT}")
        ckpt = torch.load(cfg.RESUME_CHECKPOINT, map_location=cfg.DEVICE, weights_only=False)

        raw_model.load_state_dict(ckpt['model_state_dict'])
        if 'ema_state_dict' in ckpt:
            ema.load_state_dict(ckpt['ema_state_dict'])
        if 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            for pg in optimizer.param_groups:
                pg['lr'] = cfg.LEARNING_RATE
                pg['initial_lr'] = cfg.LEARNING_RATE
        if 'scaler_state_dict' in ckpt:
            scaler.load_state_dict(ckpt['scaler_state_dict'])

        start_epoch = ckpt.get('epoch', 0) + 1
        best_loss = ckpt.get('best_loss', float('inf'))

        remaining = cfg.EPOCHS - start_epoch + 1
        lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=remaining)

        if is_main_process():
            print(f"  Resumed at epoch {start_epoch}, best_loss={best_loss:.4f}")
        del ckpt
        torch.cuda.empty_cache()

    # ═══════════════════════════════════════════════════════════════
    # TRAINING LOOP
    # ═══════════════════════════════════════════════════════════════
    if is_main_process():
        print(f"\n🚀 Training for {cfg.EPOCHS - start_epoch + 1} epochs ({start_epoch}→{cfg.EPOCHS})")
        print(f"  AMP dtype:      {cfg.AMP_DTYPE}")
        print(f"  GradScaler:     {'ON' if use_scaler else 'OFF'}")
        print(f"  CFG dropout:    {p_uncond * 100}%")
        print(f"  Timestep dist:  {cfg.TIMESTEP_SAMPLING}")

    epoch_times = []


    for epoch in range(start_epoch, cfg.EPOCHS + 1):

        # ── Time Guard ──
        elapsed_h = (time.time() - TRAINING_START_TIME) / 3600
        remaining_h = cfg.MAX_TRAINING_HOURS - elapsed_h
        if remaining_h < (cfg.GENERATION_BUDGET_MIN / 60):
            stop_reason = f"time_guard ({elapsed_h:.1f}h)"
            if is_main_process():
                print(f"\n⏰ TIME GUARD: {elapsed_h:.1f}h elapsed. Stopping.")
            break

        model.train()
        epoch_loss = 0.0
        num_batches = 0
        epoch_start = time.time()

        if IS_DDP and hasattr(train_sampler, 'set_epoch'):
            train_sampler.set_epoch(epoch)

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.EPOCHS}",
                    disable=not is_main_process())

        for batch_images, batch_labels in pbar:
            batch_images = batch_images.to(cfg.DEVICE, non_blocking=True)
            batch_labels = batch_labels.to(cfg.DEVICE, non_blocking=True)
            B = batch_images.shape[0]

            # ── CFG: randomly drop class labels ──
            mask = torch.rand(B, device=cfg.DEVICE) < p_uncond
            batch_labels_cfg = batch_labels.clone()
            batch_labels_cfg[mask] = cfg.NUM_CLASSES  # "unconditional" label

            # ══════════════════════════════════════════════════════
            # THIS IS THE CORE DIFFERENCE FROM DDPM
            # ══════════════════════════════════════════════════════

            # 1. Sample continuous timesteps t ∈ [0, 1]
            #    (DDPM uses discrete t ∈ {0, ..., 999})
            t = scheduler.sample_timesteps(
                B, method=cfg.TIMESTEP_SAMPLING,
                mean=cfg.LOGIT_NORMAL_MEAN, std=cfg.LOGIT_NORMAL_STD
            )

            # 2. Create noisy images via straight-line interpolation
            #    (DDPM uses x_t = √ᾱ·x0 + √(1-ᾱ)·ε)
            noise = torch.randn_like(batch_images)
            x_noisy, _, velocity_target = scheduler.add_noise(batch_images, t, noise)

            # 3. Predict velocity, compute loss
            #    (DDPM predicts noise: loss = MSE(ε̂, ε))
            with torch.amp.autocast('cuda', dtype=cfg.AMP_DTYPE, enabled=cfg.USE_AMP):
                v_pred = model(x_noisy, t, batch_labels_cfg)
                loss = F.mse_loss(v_pred, velocity_target)

            # ══════════════════════════════════════════════════════
            # END OF CORE DIFFERENCE — rest is identical to DDPM
            # ══════════════════════════════════════════════════════

            # Backprop
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            ema.update(raw_model)

            epoch_loss += loss.item()
            num_batches += 1
            if is_main_process():
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        # ── Epoch Summary ──
        epoch_time = time.time() - epoch_start
        epoch_times.append(epoch_time)
        avg_loss = epoch_loss / max(num_batches, 1)
        train_losses.append(avg_loss)
        lr_scheduler.step()

        elapsed_h = (time.time() - TRAINING_START_TIME) / 3600

        if is_main_process():
            eta_min = (cfg.EPOCHS - epoch) * np.mean(epoch_times) / 60
            print(f"Epoch {epoch}/{cfg.EPOCHS} | Loss: {avg_loss:.4f} | "
                  f"Best: {best_loss:.4f} | {epoch_time:.1f}s | "
                  f"Elapsed: {elapsed_h:.1f}h | ETA: {eta_min:.0f}m | "
                  f"LR: {lr_scheduler.get_last_lr()[0]:.6f}")

            # W&B logging
            if cfg.USE_WANDB and HAS_WANDB:
                log_dict = {
                    "train/loss": avg_loss,
                    "train/best_loss": min(best_loss, avg_loss),
                    "train/learning_rate": lr_scheduler.get_last_lr()[0],
                    "train/epoch_time_s": epoch_time,
                    "train/eta_minutes": eta_min,
                    "train/elapsed_hours": elapsed_h,
                    "train/epoch": epoch,
                }
                if torch.cuda.is_available():
                    log_dict["gpu/memory_allocated_gb"] = torch.cuda.memory_allocated() / 1024**3
                    log_dict["gpu/memory_reserved_gb"] = torch.cuda.memory_reserved() / 1024**3
                wandb.log(log_dict)

        # ── Save Best Model ──
        if avg_loss < best_loss:
            best_loss = avg_loss
            if is_main_process():
                save_path = os.path.join(cfg.OUTPUT_DIR, "best_rf_model.pt")
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': raw_model.state_dict(),
                    'ema_state_dict': ema.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scaler_state_dict': scaler.state_dict(),
                    'best_loss': best_loss,
                    'train_losses': train_losses,
                    'config': {
                        'method': 'rectified_flow',
                        'img_size': cfg.IMG_SIZE,
                        'base_channels': cfg.BASE_CHANNELS,
                        'num_classes': cfg.NUM_CLASSES,
                        'timestep_sampling': cfg.TIMESTEP_SAMPLING,
                    },
                }, save_path)
                print(f"  ★ Saved best model (loss={best_loss:.4f})")

        # ── Periodic Checkpoint (overwrite to save disk — matches DDPM) ──
        if epoch % cfg.SAVE_EVERY == 0 and is_main_process():
            torch.save({
                'epoch': epoch,
                'model_state_dict': raw_model.state_dict(),
                'ema_state_dict': ema.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'best_loss': best_loss,
                'train_losses': train_losses,
            }, os.path.join(cfg.OUTPUT_DIR, 'latest_checkpoint.pt'))
            print(f"  💾 Checkpoint saved (overwrite): epoch {epoch}")

        # ── Sample Images ──
        if epoch % cfg.SAMPLE_EVERY == 0 and is_main_process():
            _generate_samples(ema.shadow, scheduler, epoch)

        if IS_DDP:
            dist.barrier()

    # ── Final Save ──
    if is_main_process():
        final_path = os.path.join(cfg.OUTPUT_DIR, "final_checkpoint.pt")
        torch.save({
            'epoch': epoch,
            'model_state_dict': raw_model.state_dict(),
            'ema_state_dict': ema.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_loss': best_loss,
            'train_losses': train_losses,
            'stop_reason': stop_reason,
        }, final_path)

        print(f"\n{'='*60}")
        print(f"TRAINING COMPLETE")
        print(f"  Stop reason:  {stop_reason}")
        print(f"  Final epoch:  {epoch}")
        print(f"  Best loss:    {best_loss:.4f}")
        print(f"  Total time:   {(time.time() - TRAINING_START_TIME) / 3600:.1f}h")
        print(f"  Best model:   {os.path.join(cfg.OUTPUT_DIR, 'best_rf_model.pt')}")
        print(f"{'='*60}")

        # Plot training loss
        _plot_loss(train_losses)

        if cfg.USE_WANDB and HAS_WANDB:
            wandb.finish()

    return ema, scheduler


def _generate_samples(model, scheduler, epoch, num_per_class=4):
    """Generate sample images for visual inspection during training."""
    model.eval()
    for class_idx, class_name in cfg.IDX_TO_CLASS.items():
        labels = torch.full((num_per_class,), class_idx, device=cfg.DEVICE, dtype=torch.long)
        shape = (num_per_class, cfg.IMG_CHANNELS, cfg.IMG_SIZE, cfg.IMG_SIZE)

        samples = scheduler.sample_euler(
            model, shape, labels,
            num_steps=cfg.EULER_STEPS,
            cfg_scale=cfg.CFG_SCALE,
            num_classes=cfg.NUM_CLASSES
        )

        # Save as grid
        samples = (samples.clamp(-1, 1) + 1) / 2  # [-1,1] → [0,1]
        import torchvision.utils as vutils  # lazy import for Kaggle compat
        grid = vutils.make_grid(samples, nrow=num_per_class, padding=2)
        grid_path = os.path.join(cfg.OUTPUT_DIR, f"samples_epoch{epoch}_{class_name}.png")
        vutils.save_image(grid, grid_path)

        if cfg.USE_WANDB and HAS_WANDB:
            wandb.log({f"samples/{class_name}": wandb.Image(grid_path)})

    model.train()


def _plot_loss(train_losses):
    """Plot and save training loss curve."""
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(train_losses, label='Training Loss', linewidth=1)
        ax.set(xlabel='Epoch', ylabel='MSE Loss (velocity)',
               title='Rectified Flow Training Loss')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plot_path = os.path.join(cfg.OUTPUT_DIR, "training_loss.png")
        plt.savefig(plot_path, dpi=150)
        plt.close()

        if cfg.USE_WANDB and HAS_WANDB:
            wandb.log({"loss_curve": wandb.Image(plot_path)})
    except ImportError:
        print("matplotlib not available — skipping loss plot")


# (train.py main block removed — see unified main block at end of file)


# ═══════════════════════════════════════════════════════════════════════
# PART 6: STANDALONE GENERATION
# ═══════════════════════════════════════════════════════════════════════






def generate_synthetic_images(checkpoint_path, output_dir=None,
                              num_benign=300, num_normal=100,
                              euler_steps=30, cfg_scale=3.0,
                              batch_size=16, use_midpoint=False):
    """
    Generate synthetic CT images using a trained Rectified Flow model.
    Distributed across all available GPUs — each rank generates its share.
    
    Args:
        checkpoint_path: Path to .pt checkpoint file
        output_dir: Where to save images (default: cfg.SYNTHETIC_DIR)
        num_benign: Number of benign images to generate
        num_normal: Number of normal images to generate
        euler_steps: Number of ODE solver steps
        cfg_scale: Classifier-free guidance scale
        batch_size: Generation batch size
        use_midpoint: Use midpoint solver (2nd order, better quality)
    """
    device = cfg.DEVICE
    output_dir = output_dir or cfg.SYNTHETIC_DIR

    if is_main_process():
        print(f"\n{'='*60}")
        print(f"GENERATION — Rectified Flow (Distributed: {WORLD_SIZE} GPUs)")
        print(f"  Checkpoint:    {checkpoint_path}")
        print(f"  Output:        {output_dir}")
        print(f"  Resolution:    {cfg.IMG_SIZE}×{cfg.IMG_SIZE}")
        print(f"  Solver:        {'Midpoint (2nd order)' if use_midpoint else 'Euler (1st order)'}")
        print(f"  Steps:         {euler_steps}")
        print(f"  CFG scale:     {cfg_scale}")
        print(f"  Benign:        {num_benign}")
        print(f"  Normal:        {num_normal}")
        print(f"{'='*60}")

    # ── Load Model on this rank's GPU ──
    model = ConditionalUNet(
        img_channels=cfg.IMG_CHANNELS,
        base_channels=cfg.BASE_CHANNELS,
        time_emb_dim=cfg.TIME_EMB_DIM,
        num_classes=cfg.NUM_CLASSES,
        class_emb_dim=cfg.CLASS_EMB_DIM,
    ).to(device)

    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)

    # Try loading EMA weights first (better quality), fallback to model weights
    if 'ema_state_dict' in ckpt:
        model.load_state_dict(ckpt['ema_state_dict'])
        if is_main_process():
            print("✅ Loaded EMA weights (best quality)")
    else:
        model.load_state_dict(ckpt['model_state_dict'])
        if is_main_process():
            print("✅ Loaded model weights")

    model.eval()
    if is_main_process():
        print(f"  Epoch: {ckpt.get('epoch', '?')} | Loss: {ckpt.get('best_loss', '?')}")
    del ckpt
    torch.cuda.empty_cache()

    # ── Scheduler ──
    scheduler = RectifiedFlowScheduler(sigma_min=cfg.SIGMA_MIN, device=device)

    # ── Choose solver ──
    sample_fn = scheduler.sample_midpoint if use_midpoint else scheduler.sample_euler

    # ── Distribute generation across ranks ──
    generation_plan = {
        0: ("Benign", num_benign),
        2: ("Normal", num_normal),
    }

    for class_idx, (class_name, total_images) in generation_plan.items():
        if total_images <= 0:
            continue

        class_dir = os.path.join(output_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)

        # Split work across ranks: rank i generates images [start_idx, end_idx)
        per_rank = total_images // WORLD_SIZE
        remainder = total_images % WORLD_SIZE
        # First 'remainder' ranks get one extra image
        start_idx = RANK * per_rank + min(RANK, remainder)
        my_count = per_rank + (1 if RANK < remainder else 0)

        if is_main_process():
            print(f"\n🎨 Generating {total_images} {class_name} images across {WORLD_SIZE} GPUs...")
        print(f"  [Rank {RANK}] Generating {my_count} {class_name} images (idx {start_idx}–{start_idx + my_count - 1})")

        generated = 0
        while generated < my_count:
            current_batch = min(batch_size, my_count - generated)
            labels = torch.full((current_batch,), class_idx, device=device, dtype=torch.long)
            shape = (current_batch, cfg.IMG_CHANNELS, cfg.IMG_SIZE, cfg.IMG_SIZE)

            with torch.inference_mode():
                samples = sample_fn(
                    model, shape, labels,
                    num_steps=euler_steps,
                    cfg_scale=cfg_scale,
                    num_classes=cfg.NUM_CLASSES
                )

            # Convert [-1, 1] → [0, 255] and save
            samples = (samples.clamp(-1, 1) + 1) / 2 * 255
            samples = samples.cpu().numpy().astype(np.uint8)

            for j in range(current_batch):
                img = samples[j, 0]  # Remove channel dim
                global_idx = start_idx + generated
                img_path = os.path.join(class_dir, f"{class_name.lower()}_{global_idx:04d}.png")
                Image.fromarray(img, mode='L').save(img_path)
                generated += 1

        print(f"  [Rank {RANK}] ✅ {generated} {class_name} images saved")

    # ── Sync all ranks before evaluation ──
    if IS_DDP:
        dist.barrier()

    if is_main_process():
        print(f"\n{'='*60}")
        print(f"✅ Generation complete! All images saved to: {output_dir}")
        print(f"{'='*60}")



# ═══════════════════════════════════════════════════════════════════════
# PART 7: QUALITY EVALUATION
# ═══════════════════════════════════════════════════════════════════════





# Local imports


# ═══════════════════════════════════════════════════════════════════════
# FEATURE EXTRACTION (for FID) — InceptionV3 (Standard, Paper-Comparable)
# ═══════════════════════════════════════════════════════════════════════

class InceptionV3Features(nn.Module):
    """
    Pretrained InceptionV3 feature extractor for FID computation.
    
    Standard approach used in all FID papers. Extracts 2048-dim features
    from the pool3 layer. Grayscale images are repeated to 3 channels.
    """

    def __init__(self):
        super().__init__()
        try:
            from torchvision.models import inception_v3, Inception_V3_Weights
            inception = inception_v3(weights=Inception_V3_Weights.DEFAULT)
        except Exception:
            from torchvision.models import inception_v3
            inception = inception_v3(pretrained=True)

        # Remove the final FC layer — we want pool features (2048-dim)
        self.blocks = nn.Sequential(
            inception.Conv2d_1a_3x3, inception.Conv2d_2a_3x3,
            inception.Conv2d_2b_3x3,
            nn.MaxPool2d(kernel_size=3, stride=2),
            inception.Conv2d_3b_1x1, inception.Conv2d_4a_3x3,
            nn.MaxPool2d(kernel_size=3, stride=2),
            inception.Mixed_5b, inception.Mixed_5c, inception.Mixed_5d,
            inception.Mixed_6a, inception.Mixed_6b, inception.Mixed_6c,
            inception.Mixed_6d, inception.Mixed_6e,
            inception.Mixed_7a, inception.Mixed_7b, inception.Mixed_7c,
            nn.AdaptiveAvgPool2d(output_size=(1, 1)),
        )
        # ImageNet normalization
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        # Freeze all weights
        for param in self.parameters():
            param.requires_grad = False

    def forward(self, x):
        # x is (B, 1, H, W) in [-1, 1] — convert to 3-channel [0, 1] then normalize
        x = (x + 1) / 2  # [-1,1] → [0,1]
        x = x.repeat(1, 3, 1, 1)  # grayscale → 3 channels
        # Resize to 299×299 (InceptionV3 input size)
        x = F.interpolate(x, size=(299, 299), mode='bilinear', align_corners=False)
        # ImageNet normalization
        x = (x - self.mean) / self.std
        # Extract features
        features = self.blocks(x)
        return features.flatten(1)  # (B, 2048)


class ImageFolderDataset(Dataset):
    """Load images from a folder as grayscale tensors in [-1, 1]."""

    def __init__(self, folder, img_size=224):
        self.paths = sorted([
            p for p in Path(folder).rglob("*")
            if p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
        ])
        self.img_size = img_size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('L')
        img = img.resize((self.img_size, self.img_size))
        img = np.array(img, dtype=np.float32) / 255.0
        img = img * 2.0 - 1.0  # [0,1] → [-1,1]
        return torch.FloatTensor(img).unsqueeze(0)


# ═══════════════════════════════════════════════════════════════════════
# FID COMPUTATION
# ═══════════════════════════════════════════════════════════════════════

# Cache the feature extractor so it's only loaded once
_inception_cache = {}

def _get_inception(device):
    """Get or create cached InceptionV3 feature extractor."""
    if device not in _inception_cache:
        print("  Loading InceptionV3 feature extractor...")
        _inception_cache[device] = InceptionV3Features().to(device)
    return _inception_cache[device]


def extract_features(dataloader, feature_extractor, device):
    """Extract features from all images in a dataloader."""
    all_features = []
    feature_extractor.eval()
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting features"):
            if isinstance(batch, (list, tuple)):
                batch = batch[0]
            features = feature_extractor(batch.to(device))
            all_features.append(features.cpu().numpy())
    return np.concatenate(all_features, axis=0)


def calculate_fid(mu1, sigma1, mu2, sigma2):
    """
    Calculate Fréchet Inception Distance.
    
    FID = ||μ₁ - μ₂||² + Tr(Σ₁ + Σ₂ - 2(Σ₁Σ₂)^½)
    
    Lower = Better. FID < 50 is generally good for medical images.
    """
    from scipy.linalg import sqrtm

    diff = mu1 - mu2
    covmean, _ = sqrtm(sigma1 @ sigma2, disp=False)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return float(fid)


def compute_fid(real_dir, synthetic_dir, device, img_size=224, batch_size=32):
    """
    Compute FID between real and synthetic image directories.
    Uses pretrained InceptionV3 — standard, paper-comparable FID.
    
    Returns:
        fid_score: float (lower is better)
    """
    # Create datasets
    real_dataset = ImageFolderDataset(real_dir, img_size)
    synth_dataset = ImageFolderDataset(synthetic_dir, img_size)

    if len(real_dataset) == 0 or len(synth_dataset) == 0:
        print(f"⚠️ Empty dataset: real={len(real_dataset)}, synthetic={len(synth_dataset)}")
        return float('inf')

    real_loader = DataLoader(real_dataset, batch_size=batch_size, shuffle=False)
    synth_loader = DataLoader(synth_dataset, batch_size=batch_size, shuffle=False)

    # Pretrained InceptionV3 feature extractor (cached)
    feat_ext = _get_inception(device)

    # Extract features
    real_features = extract_features(real_loader, feat_ext, device)
    synth_features = extract_features(synth_loader, feat_ext, device)

    # Compute statistics
    mu_real = np.mean(real_features, axis=0)
    sigma_real = np.cov(real_features, rowvar=False)
    mu_synth = np.mean(synth_features, axis=0)
    sigma_synth = np.cov(synth_features, rowvar=False)

    # Regularize covariance to avoid singular matrix
    eps = 1e-6
    sigma_real += eps * np.eye(sigma_real.shape[0])
    sigma_synth += eps * np.eye(sigma_synth.shape[0])

    # Compute FID
    fid = calculate_fid(mu_real, sigma_real, mu_synth, sigma_synth)
    return fid


# ═══════════════════════════════════════════════════════════════════════
# NEAREST NEIGHBOR ANALYSIS (Memorization Check)
# ═══════════════════════════════════════════════════════════════════════

def nearest_neighbor_analysis(real_dir, synthetic_dir, device, img_size=224,
                               num_samples=50, batch_size=32):
    """
    Check if synthetic images are just copies of training data.
    
    Computes L2 distance between each synthetic image and its nearest
    real image neighbor. If distances are very small, the model memorized.
    
    Returns:
        avg_min_distance: average nearest-neighbor distance
        min_distances: list of all nearest-neighbor distances
    """
    real_dataset = ImageFolderDataset(real_dir, img_size)
    synth_dataset = ImageFolderDataset(synthetic_dir, img_size)

    if len(real_dataset) == 0 or len(synth_dataset) == 0:
        return float('inf'), []

    # Load all real images into memory
    real_images = []
    for i in range(min(len(real_dataset), 500)):
        real_images.append(real_dataset[i])
    real_tensor = torch.stack(real_images).to(device)  # (N_real, 1, H, W)

    # For each synthetic image, find nearest real neighbor
    min_distances = []
    num_check = min(num_samples, len(synth_dataset))

    for i in tqdm(range(num_check), desc="Nearest neighbor check"):
        synth_img = synth_dataset[i].unsqueeze(0).to(device)  # (1, 1, H, W)
        # L2 distance to all real images
        dists = torch.norm(
            real_tensor.flatten(1) - synth_img.flatten(1), dim=1
        )
        min_distances.append(dists.min().item())

    avg_dist = np.mean(min_distances) if min_distances else float('inf')
    return avg_dist, min_distances


# ═══════════════════════════════════════════════════════════════════════
# T-SNE VISUALIZATION
# ═══════════════════════════════════════════════════════════════════════

def plot_tsne(real_dir, synthetic_dir, device, img_size=224, batch_size=32,
              save_path=None):
    """
    Generate t-SNE plot comparing real vs synthetic image distributions.
    Uses InceptionV3 features for meaningful comparison.
    """
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        from sklearn.manifold import TSNE
    except ImportError:
        print("⚠️ matplotlib or sklearn not available — skipping t-SNE")
        return

    real_dataset = ImageFolderDataset(real_dir, img_size)
    synth_dataset = ImageFolderDataset(synthetic_dir, img_size)

    real_loader = DataLoader(real_dataset, batch_size=batch_size, shuffle=False)
    synth_loader = DataLoader(synth_dataset, batch_size=batch_size, shuffle=False)

    # Use InceptionV3 for meaningful features
    feat_ext = _get_inception(device)

    real_features = extract_features(real_loader, feat_ext, device)
    synth_features = extract_features(synth_loader, feat_ext, device)

    # Combine and run t-SNE
    all_features = np.concatenate([real_features, synth_features])

    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(all_features) - 1))
    embeddings = tsne.fit_transform(all_features)

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    n_real = len(real_features)
    ax.scatter(embeddings[:n_real, 0], embeddings[:n_real, 1],
               c='blue', alpha=0.6, label='Real', s=20)
    ax.scatter(embeddings[n_real:, 0], embeddings[n_real:, 1],
               c='red', alpha=0.6, label='Synthetic (RF)', s=20)
    ax.legend(fontsize=12)
    ax.set_title('t-SNE: Real vs Rectified Flow Synthetic (InceptionV3)', fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    save_path = save_path or os.path.join(cfg.OUTPUT_DIR, "tsne_real_vs_rf.png")
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  📊 t-SNE saved: {save_path}")


# ═══════════════════════════════════════════════════════════════════════
# MAIN EVALUATION
# ═══════════════════════════════════════════════════════════════════════

def run_evaluation(synthetic_dir=None):
    """Run complete quality evaluation suite."""
    device = cfg.DEVICE
    synthetic_dir = synthetic_dir or cfg.SYNTHETIC_DIR

    print(f"\n{'='*60}")
    print(f"QUALITY EVALUATION")
    print(f"{'='*60}")

    # Find real data directories
    real_root = cfg.DATASET_ROOT

    results = {}

    for class_name in ["Benign", "Normal"]:
        # Handle "Bengin" typo in dataset folder name
        search_terms = [class_name.lower()]
        if class_name.lower() == "benign":
            search_terms.append("bengin")  # dataset typo
        matches = [c for c in cfg.CLASSES if any(s in c.lower() for s in search_terms)]
        if not matches:
            print(f"  ⚠️ No matching class folder for '{class_name}' in {cfg.CLASSES}")
            continue
        real_class_dir = os.path.join(real_root, matches[0])
        synth_class_dir = os.path.join(synthetic_dir, class_name)

        if not os.path.exists(synth_class_dir):
            print(f"  ⚠️ No synthetic {class_name} images found")
            continue

        synth_count = len(list(Path(synth_class_dir).glob("*.png")))
        if synth_count == 0:
            continue

        print(f"\n── {class_name} ──")
        print(f"  Synthetic images: {synth_count}")

        # FID
        fid = compute_fid(real_class_dir, synth_class_dir, device,
                          img_size=cfg.IMG_SIZE)
        print(f"  FID: {fid:.2f}")
        results[f"fid_{class_name.lower()}"] = fid

        # FID interpretation
        if fid < 20:
            quality = "🟢 Excellent"
        elif fid < 50:
            quality = "🟢 Good"
        elif fid < 100:
            quality = "🟡 Acceptable"
        elif fid < 200:
            quality = "🟠 Mediocre"
        else:
            quality = "🔴 Poor"
        print(f"  Quality: {quality}")

        # Nearest neighbor (memorization check)
        avg_nn, _ = nearest_neighbor_analysis(
            real_class_dir, synth_class_dir, device, img_size=cfg.IMG_SIZE
        )
        print(f"  Avg NN distance: {avg_nn:.4f}")
        if avg_nn < 0.1:
            print(f"  ⚠️ WARNING: Very low NN distance — possible memorization!")
        results[f"nn_dist_{class_name.lower()}"] = avg_nn

        # t-SNE
        tsne_path = os.path.join(cfg.OUTPUT_DIR, f"tsne_{class_name.lower()}.png")
        plot_tsne(real_class_dir, synth_class_dir, device,
                  img_size=cfg.IMG_SIZE, save_path=tsne_path)

    # Save results
    results_path = os.path.join(cfg.OUTPUT_DIR, "quality_scores.txt")
    with open(results_path, 'w') as f:
        f.write("Rectified Flow — Quality Evaluation Results\n")
        f.write("=" * 50 + "\n")
        for k, v in results.items():
            f.write(f"{k}: {v:.4f}\n")
    print(f"\n📄 Results saved: {results_path}")

    print(f"\n{'='*60}")
    print(f"✅ Evaluation complete")
    print(f"{'='*60}")

    return results



# ═══════════════════════════════════════════════════════════════════════
# MAIN — Run Everything
# ═══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    try:
        # Train the model
        ema, scheduler = train()

        # Generate synthetic images (distributed across ALL GPUs)
        generate_synthetic_images(
            checkpoint_path=os.path.join(cfg.OUTPUT_DIR, "best_rf_model.pt"),
            output_dir=cfg.SYNTHETIC_DIR,
            num_benign=cfg.NUM_SYNTHETIC_BENIGN,
            num_normal=cfg.NUM_SYNTHETIC_NORMAL,
            euler_steps=cfg.EULER_STEPS,
            cfg_scale=cfg.CFG_SCALE,
        )
        # barrier is inside generate_synthetic_images

        # Evaluate on rank 0 only (reads from shared filesystem)
        if is_main_process():
            run_evaluation(synthetic_dir=cfg.SYNTHETIC_DIR)

    except Exception as e:
        if is_main_process():
            print(f"\n Failed: {e}")
            import traceback
            traceback.print_exc()
    finally:
        cleanup_distributed()


Writing rectified_flow_ct_augmentation.py


In [3]:
!torchrun --nproc_per_node=2 rectified_flow_ct_augmentation.py

W0227 00:02:56.762000 57 torch/distributed/run.py:803] 
W0227 00:02:56.762000 57 torch/distributed/run.py:803] *****************************************
W0227 00:02:56.762000 57 torch/distributed/run.py:803] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0227 00:02:56.762000 57 torch/distributed/run.py:803] *****************************************
/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/c